## Data ophalen

In [8]:
import requests
import pandas as pd
import time

# Alle punten over Blasco Ibáñez
punten = {
    "blasco_01_west": (39.4750, -0.3670),
    "blasco_02":      (39.4745, -0.3645),
    "blasco_03":      (39.4740, -0.3620),
    "blasco_04":      (39.4735, -0.3595),
    "blasco_05":      (39.4730, -0.3570),
    "blasco_06":      (39.4725, -0.3545),
    "blasco_07":      (39.4720, -0.3520),
    "blasco_08":      (39.4715, -0.3490),
    "blasco_09":      (39.4705, -0.3460),
    "blasco_10_oost": (39.4695, -0.3430)
}

url = "https://archive-api.open-meteo.com/v1/archive"

namen = list(punten.keys())
latitudes = [punten[x][0] for x in namen]
longitudes = [punten[x][1] for x in namen]

params = {
    "latitude": ",".join(map(str, latitudes)),
    "longitude": ",".join(map(str, longitudes)),
    "start_date": "2015-01-01",
    "end_date": "2025-12-31",

    "hourly": ",".join([
        "temperature_2m",
        "relative_humidity_2m",
        "apparent_temperature",
        "precipitation",
        "wind_speed_10m",
        "wind_direction_10m",
        "shortwave_radiation"
    ]),

    "timezone": "Europe/Madrid"
}

print("Data ophalen voor alle locaties...")

response = requests.get(url, params=params, timeout=120)

print("Status:", response.status_code)

response.raise_for_status()

data = response.json()

print("Data ontvangen!")

Data ophalen voor alle locaties...
Status: 200
Data ontvangen!


In [9]:
alle_data = []

for i, naam in enumerate(namen):

    locatie_data = data[i]["hourly"]

    df = pd.DataFrame(locatie_data)

    df["locatie"] = naam
    df["latitude"] = latitudes[i]
    df["longitude"] = longitudes[i]

    alle_data.append(df)

resultaat = pd.concat(alle_data, ignore_index=True)

resultaat["time"] = pd.to_datetime(resultaat["time"])

print("Aantal rijen:", len(resultaat))
print("Aantal locaties:", resultaat["locatie"].nunique())

resultaat.to_csv(
    "blasco_ibanez_weather_2015_2025.csv",
    index=False
)

print("CSV opgeslagen!")

Aantal rijen: 964320
Aantal locaties: 10
CSV opgeslagen!


## Punten visualiseren

In [10]:
import folium

kaart = folium.Map(
    location=[sum(latitudes) / len(latitudes), sum(longitudes) / len(longitudes)],
    zoom_start=15,
    tiles="OpenStreetMap"
)

coördinaten = []

for naam, (lat, lon) in punten.items():
    coördinaten.append([lat, lon])

    folium.Marker(
        location=[lat, lon],
        popup=f"{naam}<br>Latitude: {lat}<br>Longitude: {lon}",
        tooltip=naam,
        icon=folium.Icon(color="blue", icon="info-sign")
    ).add_to(kaart)

folium.PolyLine(
    coördinaten,
    color="red",
    weight=3,
    opacity=0.8
).add_to(kaart)

kaart